In [ ]:
import cv2
import numpy as np
import torch

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# Paths from official README
checkpoint = "sam2/checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"  # inside sam2 repo

# Build predictor
predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint))

# Load any image (for a sanity test you can use any .jpg/.png)
img_path = "notebooks/assets/example.jpg"  # replace with your own image path
image_bgr = cv2.imread(img_path)
assert image_bgr is not None, f"Could not read {img_path}"
image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# Example prompt: a box in XYXY format
h, w = image.shape[:2]
box = np.array([w*0.25, h*0.25, w*0.75, h*0.75], dtype=np.float32)

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
    predictor.set_image(image)
    masks, scores, _ = predictor.predict(box=box)

print("masks shape:", masks.shape, "scores:", scores[:3])